In [1]:
##Base (python 3.12.7)
## Install required library for RDF processing
! pip install rdflib

import rdflib
from rdflib import Graph, Literal, BNode, Namespace, URIRef
from rdflib.namespace import RDF, RDFS, OWL, XSD
import pandas as pd

## Initialize graph and load semantic data
g = Graph()

## Set up namespace matching my ontology
MOVIE = Namespace("http://www.semanticweb.org/legion/ontologies/2026/0/untitled-ontology-5/")
g.bind("", MOVIE)

## Import RDF dataset
print("Importing RDF triples...")
g.parse("../Data_collection/movie_ontology_rdf.ttl", format="turtle")
print(f"Success! Total triples loaded: {len(g)}")

# ============================================
# Query 1: List all films
# ============================================
print("\n" + "="*50)
print("Query 1: Display all movie entries")
print("="*50)

q1 = """
PREFIX : <http://www.semanticweb.org/legion/ontologies/2026/0/untitled-ontology-5/>

SELECT ?movie ?title ?year
WHERE {
    ?movie a :Movie .
    ?movie :title ?title .
    OPTIONAL { ?movie :releaseYear ?year . }
}
ORDER BY ?title 
LIMIT 10
"""

results = g.query(q1)
for row in results:
    print(f"Film: {row.title} (Released: {row.year})")

# ============================================
# Query 2: Filter by director (dynamic version)
# ============================================
print("\n" + "="*50)
print("Query 2: Locate films by specific director")
print("="*50)

def find_by_director(director_name):
    """Retrieve movies directed by a particular person"""
    
    director_uri = director_name.replace(' ', '_')
    
    query = f"""
    PREFIX : <http://www.semanticweb.org/legion/ontologies/2026/0/untitled-ontology-5/>
    
    SELECT ?movie ?title ?year ?rating
    WHERE {{
        ?movie a :Movie .
        ?movie :title ?title .
        ?movie :directedBy :{director_uri} .
        OPTIONAL {{ ?movie :releaseYear ?year . }}
        OPTIONAL {{ ?movie :rating ?rating . }}
    }}
    ORDER BY DESC(?rating)
    """
    
    return g.query(query)

# Example: Christopher Nolan's works
results = find_by_director("Christopher_Nolan")
for row in results:
    print(f"{row.title} ({row.year}) - Score: {row.rating}")

# ============================================
# Query 3: Filter by cast member (enhanced with multi-keyword)
# ============================================
print("\n" + "="*50)
print("Query 3: Check  by actor or actress name")
print("="*50)
 
## Install required library 
! pip install python-Levenshtein 

import Levenshtein
from difflib import SequenceMatcher

_actors_cache = None # Cache for actor list to avoid repeated queries

def get_all_actors():
    """Get list of all actor names (with caching)"""
    global _actors_cache
    if _actors_cache is None:
        query = """
        PREFIX : <http://www.semanticweb.org/legion/ontologies/2026/0/untitled-ontology-5/>
        
        SELECT DISTINCT ?actorName
        WHERE {
            ?actor a :Actor_Actress .
            ?actor :name ?actorName .
        }
        """
        results = g.query(query)
        _actors_cache = [str(row.actorName) for row in results]
        print(f"  Loaded {len(_actors_cache)} actors for fuzzy search")
    return _actors_cache

def find_similar_actors(input_name, threshold=0.5):
    """Find similar actors using edit distance"""
    actors = get_all_actors()
    input_lower = input_name.lower().strip()
    
    if len(input_lower) < 2:
        return []
    
    matches = []
    for actor in actors:
        actor_lower = actor.lower()
        
        # Calculate Levenshtein similarity ratio
        lev_ratio = Levenshtein.ratio(input_lower, actor_lower)
        
        # Add to matches if above threshold
        if lev_ratio > threshold:
            matches.append((actor, lev_ratio))
    
    # Sort by similarity score
    matches.sort(key=lambda x: x[1], reverse=True)
    return matches[:5]  # Return top 5 most similar

def find_by_actor(actor_name):
    """Search for movies starring a particular performer with fuzzy matching"""
    
    # 1. First try exact keyword matching
    import re
    clean_input = re.sub(r'\s+', ' ', actor_name.lower().strip())
    keywords = [k for k in clean_input.split() if len(k) > 2]
    
    if keywords:
        conditions = [f'CONTAINS(LCASE(?actorName), "{k}")' for k in keywords]
        filter_text = ' && '.join(conditions)
        
        exact_query = f"""
        PREFIX : <http://www.semanticweb.org/legion/ontologies/2026/0/untitled-ontology-5/>
        
        SELECT DISTINCT ?movie ?title ?year ?rating ?actorName
        WHERE {{
            ?actor a :Actor_Actress .
            ?actor :name ?actorName .
            FILTER({filter_text})
            
            ?actor :actedIn ?movie .
            ?movie :title ?title .
            OPTIONAL {{ ?movie :releaseYear ?year . }}
            OPTIONAL {{ ?movie :rating ?rating . }}
        }}
        ORDER BY DESC(?year)
        LIMIT 20
        """
        
        exact_results = list(g.query(exact_query))
        if exact_results:
            return exact_results
    
    # 2. If no exact matches, try fuzzy matching with edit distance
    print(f"   No exact matches for '{actor_name}', using fuzzy search") ##Technology 
    similar = find_similar_actors(actor_name)
    
    if not similar:
        print("   No similar actors found.")
        return []
    
    # Display found similar actors
    print(f"   Found similar actors:")
    for actor, score in similar[:3]:
        print(f"   • {actor} (similarity: {score:.3f})")
    
    # Use best match for query
    best_match = similar[0][0]
    print(f"   Using best match: {best_match}")
    
    fuzzy_query = f"""
    PREFIX : <http://www.semanticweb.org/legion/ontologies/2026/0/untitled-ontology-5/>
    
    SELECT DISTINCT ?movie ?title ?year ?rating ?actorName
    WHERE {{
        ?actor a :Actor_Actress .
        ?actor :name ?actorName .
        FILTER(CONTAINS(LCASE(?actorName), LCASE("{best_match}")))
        
        ?actor :actedIn ?movie .
        ?movie :title ?title .
        OPTIONAL {{ ?movie :releaseYear ?year . }}
        OPTIONAL {{ ?movie :rating ?rating . }}
    }}
    ORDER BY DESC(?year)
    LIMIT 20
    """
    
    return g.query(fuzzy_query)

# Example: Leonardo DiCaprio filmography
results = find_by_actor("di caprio")
for row in results:
    print(f"{row.title} ({row.year}) - Starring: {row.actorName} - Rating: {row.rating}")

# Test: Robert Downey Jr. variations
results = find_by_actor("robet downey")
for row in results:
    print(f"{row.title} ({row.year}) - Starring: {row.actorName} - Rating: {row.rating}")
    
# ============================================
# Query 4: Filter by category
# ============================================
print("\n" + "="*50)
print("Query 4: Identify films by genre")
print("="*50)

def find_by_genre(genre_type):
    """Locate movies belonging to a specific category"""
    
    query = f"""
    PREFIX : <http://www.semanticweb.org/legion/ontologies/2026/0/untitled-ontology-5/>
    
    SELECT ?movie ?title ?year ?rating ?genreType
    WHERE {{
        ?genre a :Genre .
        ?genre :GenreName ?genreType .
        FILTER(CONTAINS(LCASE(?genreType), LCASE("{genre_type}")))
        
        ?movie :hasGenre ?genre .
        ?movie :title ?title .
        OPTIONAL {{ ?movie :releaseYear ?year . }}
        OPTIONAL {{ ?movie :rating ?rating . }}
    }}
    ORDER BY DESC(?rating)
    LIMIT 15
    """
    
    return g.query(query)

# Example: Action movies
results = find_by_genre("action")
for row in results:
    print(f"{row.title} ({row.year}) - Category: {row.genreType} - Score: {row.rating}")

# ============================================
# Query 5: Filter by production studio
# ============================================
print("\n" + "="*50)
print("Query 5: Discover films by production company")
print("="*50)

def find_by_studio(studio_name):
    """Search for movies produced by specific company"""
    
    query = f"""
    PREFIX : <http://www.semanticweb.org/legion/ontologies/2026/0/untitled-ontology-5/>
    
    SELECT ?movie ?title ?year ?rating ?studioName
    WHERE {{
        ?studio a :Company .
        ?studio :CompanyName ?studioName .
        FILTER(CONTAINS(LCASE(?studioName), LCASE("{studio_name}")))
        
        ?movie :producedBy ?studio .
        ?movie :title ?title .
        OPTIONAL {{ ?movie :releaseYear ?year . }}
        OPTIONAL {{ ?movie :rating ?rating . }}
    }}
    ORDER BY DESC(?rating)
    LIMIT 15
    """
    
    return g.query(query)

# Example: Warner Bros productions
results = find_by_studio("warner")
for row in results:
    print(f"{row.title} ({row.year}) - Studio: {row.studioName} - Score: {row.rating}")

# ============================================
# Query 6: Multi-condition search
# ============================================
print("\n" + "="*50)
print("Query 6: Advanced filtering with multiple criteria")
print("="*50)

def advanced_filter(category=None, from_year=None, min_score=None):
    """Apply multiple filters simultaneously"""
    
    conditions = []
    if category:
        conditions.append(f'?genre :GenreName "{category}"@en .')
        conditions.append('?movie :hasGenre ?genre .')
    if from_year:
        conditions.append(f'?movie :releaseYear ?year . FILTER(?year >= {from_year})')
    if min_score:
        conditions.append(f'?movie :rating ?rating . FILTER(?rating >= {min_score})')
    
    filter_block = "\n    ".join(conditions)
    
    query = f"""
    PREFIX : <http://www.semanticweb.org/legion/ontologies/2026/0/untitled-ontology-5/>
    
    SELECT ?movie ?title ?year ?rating
    WHERE {{
        ?movie a :Movie .
        ?movie :title ?title .
        OPTIONAL {{ ?movie :releaseYear ?year . }}
        OPTIONAL {{ ?movie :rating ?rating . }}
        
        {filter_block}
    }}
    ORDER BY DESC(?rating)
    LIMIT 20
    """
    
    return g.query(query)

# Example: Action films post-2010 with high ratings
results = advanced_filter(category="Action", from_year=2010, min_score=7.0)
for row in results:
    print(f"{row.title} ({row.year}) - Score: {row.rating}")

# ============================================
# Query 7: Time-based filtering
# ============================================
print("\n" + "="*50)
print("Query 7: Retrieve films from specific period")
print("="*50)

def filter_by_period(start_year, end_year):
    """Find movies released within a year range"""
    
    query = f"""
    PREFIX : <http://www.semanticweb.org/legion/ontologies/2026/0/untitled-ontology-5/>
    
    SELECT ?movie ?title ?year ?rating
    WHERE {{
        ?movie a :Movie .
        ?movie :title ?title .
        ?movie :releaseYear ?year .
        OPTIONAL {{ ?movie :rating ?rating . }}
        
        FILTER(?year >= {start_year} && ?year <= {end_year})
    }}
    ORDER BY DESC(?year)
    LIMIT 20
    """
    
    return g.query(query)

# Example: Films from 2010-2015
results = filter_by_period(2010, 2015)
for row in results:
    print(f"{row.title} ({row.year}) - Score: {row.rating}")

# ============================================
# Comprehensive Search (Covers all fields)
# ============================================
print("\n" + "="*50)
print("Universal Search: Find matches across all categories")
print("="*50)

def comprehensive_search(term):
    """Search across titles, directors, actors, genres, and studios"""
    
    query = f"""
    PREFIX : <http://www.semanticweb.org/legion/ontologies/2026/0/untitled-ontology-5/>
    
    SELECT DISTINCT ?movie ?title ?year ?score ?matchedField ?matchedText
    WHERE {{
        {{
            # Match in film titles
            ?movie a :Movie .
            ?movie :title ?title .
            OPTIONAL {{ ?movie :releaseYear ?year . }}
            OPTIONAL {{ ?movie :rating ?score . }}
            FILTER(CONTAINS(LCASE(?title), LCASE("{term}")))
            BIND("Title" AS ?matchedField)
            BIND(?title AS ?matchedText)
        }}
        UNION
        {{
            # Match in director names
            ?movie a :Movie .
            ?movie :title ?title .
            ?movie :directedBy ?director .
            ?director :name ?directorName .
            FILTER(CONTAINS(LCASE(?directorName), LCASE("{term}")))
            OPTIONAL {{ ?movie :releaseYear ?year . }}
            OPTIONAL {{ ?movie :rating ?score . }}
            BIND("Director" AS ?matchedField)
            BIND(?directorName AS ?matchedText)
        }}
        UNION
        {{
            # Match in actor names
            ?movie a :Movie .
            ?movie :title ?title .
            ?actor :actedIn ?movie .
            ?actor :name ?actorName .
            FILTER(CONTAINS(LCASE(?actorName), LCASE("{term}")))
            OPTIONAL {{ ?movie :releaseYear ?year . }}
            OPTIONAL {{ ?movie :rating ?score . }}
            BIND("Actor" AS ?matchedField)
            BIND(?actorName AS ?matchedText)
        }}
        UNION
        {{
            # Match in genre names
            ?movie a :Movie .
            ?movie :title ?title .
            ?movie :hasGenre ?genre .
            ?genre :GenreName ?genreName .
            FILTER(CONTAINS(LCASE(?genreName), LCASE("{term}")))
            OPTIONAL {{ ?movie :releaseYear ?year . }}
            OPTIONAL {{ ?movie :rating ?score . }}
            BIND("Genre" AS ?matchedField)
            BIND(?genreName AS ?matchedText)
        }}
        UNION
        {{
            # Match in studio names
            ?movie a :Movie .
            ?movie :title ?title .
            ?movie :producedBy ?studio .
            ?studio :CompanyName ?studioName .
            FILTER(CONTAINS(LCASE(?studioName), LCASE("{term}")))
            OPTIONAL {{ ?movie :releaseYear ?year . }}
            OPTIONAL {{ ?movie :rating ?score . }}
            BIND("Studio" AS ?matchedField)
            BIND(?studioName AS ?matchedText)
        }}
    }}
    ORDER BY DESC(?score)
    LIMIT 30
    """
    
    return g.query(query)

# Example: Search for "batman"
results = comprehensive_search("batman")
for row in results:
    print(f"{row.title} ({row.year}) - Found in: {row.matchedField} - '{row.matchedText}' - Score: {row.score}")

Importing RDF triples...
Success! Total triples loaded: 7072

Query 1: Display all movie entries
Film: 2012 (Released: 2009)
Film: 47 Ronin (Released: 2013)
Film: A Christmas Carol (Released: 2009)
Film: After Earth (Released: 2013)
Film: Alexander (Released: 2004)
Film: Alice Through the Looking Glass (Released: 2016)
Film: Alice in Wonderland (Released: 2010)
Film: Angels & Demons (Released: 2009)
Film: Ant-Man (Released: 2015)
Film: Armageddon (Released: 1998)

Query 2: Locate films by specific director
The Dark Knight (2008) - Score: 8.2
Interstellar (2014) - Score: 8.1
Inception (2010) - Score: 8.1
The Dark Knight Rises (2012) - Score: 7.6
Batman Begins (2005) - Score: 7.5

Query 3: Check  by actor or actress name
The Revenant (2015) - Starring: Leonardo DiCaprio - Rating: 7.3
The Great Gatsby (2013) - Starring: Leonardo DiCaprio - Rating: 7.3
Inception (2010) - Starring: Leonardo DiCaprio - Rating: 8.1
Titanic (1997) - Starring: Leonardo DiCaprio - Rating: 7.5
   No exact matches